In [2]:
import numpy as np

rng = np.random.default_rng(7)

class STO():#SyntheticThinkingOrganoid
    def __init__(self,n_inputs):
        self.w = rng.normal(0,1,size=n_inputs) * 0.1
        self.b = 0.0
        self.mean = []
        
    def forward(self,x):
        s = x @ self.w + self.b
        return np.tanh(s) 
    
    def learn(self,x,t,lr):
        y = self.forward(x) 
        e = t - y
        blame = e * (1 - y ** 2)
        self.w += lr * blame * x
        self.b += lr * blame
        self.mean.append(abs(e))
        
    def print_weights(self):
        print(self.w)
        print(self.b)
        
    def reset_memory(self):
        self.mean = []
        
n = STO(3)
x = np.array([3,5,2])
true_w = np.array([0.5,-0.3,0.6])
print(n.forward(x))

0.09463024860292284


In [3]:
wave = 0.5 * np.sin(np.linspace(0,20,500)) 
X = []
Y = []
for i in range(len(wave) - 3):
    X.append(wave[i:i + 3])
    Y.append(wave[i + 3])

X = np.array(X)
Y = np.array(Y)

wave2 = 0.5 * np.sin(8 * np.linspace(0,20,500)) 
X2 = []
Y2 = []
for i in range(len(wave2) - 3):
    X2.append(wave2[i:i + 3])
    Y2.append(wave2[i + 3])

X2 = np.array(X2)
Y2 = np.array(Y2)

xa = rng.uniform(0.1,0.5,500)
xb = rng.uniform(-0.5,-0.1,500)
XA = xa[:,None]
XB = xb[:,None]
YA = np.full(500,0.4)
YB = np.full(500,-0.1)

print(X.shape,Y.shape)
print(X2.shape,Y2.shape)
print(XA.shape,YA.shape)
print(XB.shape,YB.shape)

(497, 3) (497,)
(497, 3) (497,)
(500, 1) (500,)
(500, 1) (500,)


In [4]:
def memory_base(seed):
    global rng
    rng = np.random.default_rng(seed)
    n = STO(1)
    for epoch in range(30):
        for i in range(400):
            n.learn(XA[i],YA[i],0.1)
    n.reset_memory()
    for i in range(400,500):
        n.learn(XA[i],YA[i],0.0)
    base_a = sum(n.mean) / len(n.mean)

    for epoch in range(30):
        for i in range(400):
            n.learn(XB[i],YB[i],0.1)
    n.reset_memory()
    for i in range(400,500):
        n.learn(XB[i],YB[i],0.0)
    base_b = sum(n.mean) / len(n.mean)

    n.reset_memory()
    for i in range(400,500):
        n.learn(XA[i],YA[i],0.0)
    base_aa = sum(n.mean) / len(n.mean)
    return base_a,base_b,base_aa

In [5]:
threshold = 0.05
gate = 1

class STO_tree():
    def __init__(self):
        self.branches = {} 
        self.b = 0.0
        self.next_id = 0
        self.exits = []
        self.rval = 0
        self.activation = 0.0
        self.mean = [] 
        self.gain = 1.0
        self.act_avg = 0.5
        self.cool = 0
        
        
    def roots(self,n_inputs):
        self.n_roots = n_inputs
        for i in range(n_inputs):
            self.exits.append(self.next_id)
            self.branches[self.next_id] = {"exit_type": 1,
            "parent": None,
            "children": [],
            "weight": 1 + rng.normal(0,0.1),
            "pos": "for later",
            "entry": 0.0,
            "exit": 0.0,
            "gain": 0.0,
            "stiff": 0.0,
            "activity": 0.5,
            "avg_blame": 0.0,
            "input_idx": i,
            "involvment": 1.0}
            self.next_id += 1
            
    def branch(self,parent,am,inputs = None):
        self.branches[parent]["exit_type"] = 0
        self.exits.remove(parent)
        for i in range(am):
            self.exits.append(self.next_id)
            self.branches[self.next_id] = {"exit_type": 1,
            "parent": parent,
            "children": [],
            "weight": 1 + rng.normal(0,0.1),
            "pos": "for later",
            "entry": 0.0,
            "exit": 0.0,
            "gain": 0.0,
            "stiff": 0.0,
            "activity": 0.5,
            "avg_blame": 0.0,
            "input_idx": (inputs[i] if inputs else self.branches[parent]["input_idx"]),
            "involvment": self.branches[parent]["involvment"] / am}
            self.branches[parent]["children"].append(self.next_id)
            self.next_id += 1
            
    def forward(self,x):
        self.rval = 0
        for a,b in enumerate(self.exits): #inputs
            self.branches[b]["entry"] = x[self.branches[b]["input_idx"]]
            self.branches[b]["exit"] = np.tanh(self.branches[b]["weight"] * self.branches[b]["entry"])
            if abs(self.branches[b]["entry"]) > threshold:
                self.branches[b]["gain"] = self.branches[b]["exit"] / self.branches[b]["entry"]
            else:
                self.branches[b]["gain"] = 1.0
                
        for be,bid in reversed([(e,self.branches[e]) for e in self.branches if self.branches[e]["exit_type"] == 0]): #paren
            val = 0
            for c in bid["children"]:
                val += self.branches[c]["exit"]
            self.branches[be]["entry"] = val / len(bid["children"])
            u = self.branches[be]["weight"] * self.branches[be]["entry"]
            self.branches[be]["exit"] = u * np.exp(0.5 - 0.5 * u ** 2)
            
            if abs(self.branches[be]["weight"]) <0.001:
                self.branches[be]["gain"] = 0
            else:
                center_entry = 1 / self.branches[be]["weight"]
                d = self.branches[be]["entry"] - center_entry
                if abs(d) > threshold:
                    self.branches[be]["gain"] = (self.branches[be]["exit"] - 1) / d
                else:
                    self.branches[be]["gain"] = 0
        for r in range(self.n_roots):
            self.rval += self.branches[r]["exit"]
        self.activation = self.rval + self.b
        y = np.tanh(self.activation)
        if abs(self.activation) > threshold:
            self.gain = y / self.activation
        else:
            self.gain = 1.0
        return y
    
    def learn(self,x,t,lr):
        y = self.forward(x) 
        self.mean.append(abs(t - y))
        if self.act_avg < 0.1:
            self.act_avg = 0.5
        self.act_avg = 0.99 * self.act_avg + 0.01 * abs(self.rval)
        blame = (t - y) / self.gain
        share = blame / (self.n_roots + 1)
        to_sprout = []
        reach = {} 
        for ro in range(self.n_roots):
            reach[ro] = 1.0
        for o in self.branches:
            if self.branches[o]["parent"] is not None:
                reach[o] = reach[self.branches[o]["parent"]] * self.branches[self.branches[o]["parent"]]["gain"]
            its_blame = share * reach[o] * self.branches[o]["involvment"]
            if abs(self.branches[o]["entry"]) > threshold and abs(self.branches[o]["exit"]) > gate * self.branches[o]["activity"]:
                change_weight = lr * its_blame * np.sign(self.branches[o]["entry"])
                if self.branches[o]["children"] != []:
                    change_weight *= (1 if abs(self.branches[o]["weight"] * self.branches[o]["entry"]) < 1 else (-1))
                self.branches[o]["weight"] += change_weight
            self.branches[o]["activity"] = 0.99 * self.branches[o]["activity"] + 0.01 * abs(self.branches[o]["exit"])
            self.branches[o]["avg_blame"] = 0.99 * self.branches[o]["avg_blame"] + 0.01 * abs(its_blame)
            if self.branches[o]["children"] == [] and self.branches[o]["avg_blame"] > 0.015:
                to_sprout.append(o)
            if self.act_avg < 0.1:
                self.branches[o]["weight"] = 1 + rng.normal(0,0.1)
                self.branches[o]["activity"] = 0.5
                self.branches[o]["avg_blame"] = 0
        self.b += lr * share
        self.cool -= 1
        for sprout_me in to_sprout:
            if len(self.branches) < 15 and self.cool <= 0 and max(self.exits,key=lambda i: self.branches[i]["avg_blame"]) == sprout_me:
                self.branch(sprout_me,2)
                self.cool = 395
        
    def reset_memory(self):
        self.mean = []           
            
            
#hope this cutie works, so lovelyyyyyyyy
tree = STO_tree()
tree.roots(2)
tree.branch(0,2)
print(tree.forward(np.array([0.5,0.4,0.3])))
for repeat in range(20):
    tree.learn(np.array([0.5, 0.4, 0.3]), 0.9, 0.1)
print(tree.forward(np.array([0.5,0.4,0.3])))
print()
for test in tree.branches:
    print("-",test)
    print(tree.branches[test]["entry"])
    print(tree.branches[test]["exit"])
    print()
print(tree.branches[3])

0.8042576136854284
0.8411825157356014

- 0
0.4832045720650221
0.738827281372782

- 1
0.4
0.41440759922938003

- 2
0.5
0.45652586301587933

- 3
0.5
0.5098832811141648

{'exit_type': 1, 'parent': 0, 'children': [], 'weight': np.float64(1.1251440649990412), 'pos': 'for later', 'entry': np.float64(0.5), 'exit': np.float64(0.5098832811141648), 'gain': np.float64(1.0197665622283296), 'stiff': 0.0, 'activity': np.float64(0.5011504532637915), 'avg_blame': np.float64(0.0018400118475169925), 'input_idx': 0, 'involvment': 0.5}


In [6]:


def memory_test(seed):
    global rng
    rng = np.random.default_rng(seed)
    test_me = STO_tree()
    test_me.roots(1)
    for epoch in range(30):
        for i in range(400):
            test_me.learn(XA[i],YA[i],0.1)
    test_me.reset_memory()
    for i in range(400,500):
        test_me.learn(XA[i],YA[i],0.0)
    test_a = sum(test_me.mean) / len(test_me.mean)
    
    worst = max(test_me.exits,key=lambda i: test_me.branches[i]["avg_blame"])
    #print(worst,"|",test_me.branches[worst]["avg_blame"],"| A -")

    for epoch in range(30):
        for i in range(400):
            test_me.learn(XB[i],YB[i],0.1)
        if epoch in (0,1,4):
            worst = max(test_me.exits,key=lambda i: test_me.branches[i]["avg_blame"])
            #print(worst,"|",test_me.branches[worst]["avg_blame"],"| B",epoch)
    test_me.reset_memory()
    for i in range(400,500):
        test_me.learn(XB[i],YB[i],0.0)
    test_b = sum(test_me.mean) / len(test_me.mean)
    
    worst = max(test_me.exits,key=lambda i: test_me.branches[i]["avg_blame"])
    #print(worst,"|",test_me.branches[worst]["avg_blame"],"| B -")

    test_me.reset_memory()
    for i in range(400,500):
        test_me.learn(XA[i],YA[i],0.0)
    test_aa = sum(test_me.mean) / len(test_me.mean)
    return test_a,test_b,test_aa

In [7]:

def output_correct(forward_result):
    correct_value = 0
    if forward_result[0] < 0:
        correct_value += 1
    if forward_result[1] > 0:
        correct_value += 1 
    if forward_result[2] > 0:
        correct_value += 1
    if forward_result[3] < 0:
        correct_value += 1
    return correct_value

def test_seed(seed):
    global rng
    rng = np.random.default_rng(seed)
    r1,r2 = 0,0
    P = np.array([(0,0),(0,1),(1,0),(1,1)])
    T = np.array([-0.5,0.5,0.5,-0.5])
    xor_test = STO_tree()
    xor_test.roots(1)
    xor_test.branch(0,2,[0,1])
    xor_base = STO(2)
    for epoch in range(2000):
        for k in rng.permutation(4):
            xor_test.learn(P[k],T[k],1.0) 
            xor_base.learn(P[k],T[k],0.1)
    forward_result = []
    for count in range(4):
        forward_result.append(xor_test.forward(P[count]))
    score = output_correct(forward_result)
    if score == 4:
        r1 += 1
    else:
        print(seed,"DEAD: ",xor_test.branches[0]["activity"],"|",xor_test.branches[0]["avg_blame"])
    forward_result = []
    for count in range(4):
        forward_result.append(xor_base.forward(P[count]))
    score = output_correct(forward_result)
    if score == 4:
        r2 += 1
    return r1,r2

In [8]:
test_score,base_score = 0,0
test_a_val,test_b_val,test_aa_val = 0,0,0
base_a_val,base_b_val,base_aa_val = 0,0,0
seed_samples = 100
for i in range(seed_samples):
    print(i,end="|")
    returned_score = test_seed(i)
    test_score += returned_score[0]
    base_score += returned_score[1]
    returned_test_val = memory_test(i)
    test_a_val += returned_test_val[0]
    test_b_val += returned_test_val[1]
    test_aa_val +=  returned_test_val[2]
    returned_base_val = memory_base(i)
    base_a_val += returned_base_val[0]
    base_b_val += returned_base_val[1]
    base_aa_val +=  returned_base_val[2]
print("\n\n",(base_b_val + base_aa_val) / (test_b_val + test_aa_val),"\nTIMES MORE EFFICIENT ON A/B")
print("\nMEMORY TEST OF TREE: \n-A:",test_a_val / seed_samples,"\n-B:",test_b_val / seed_samples,"\n-AA:",test_aa_val / seed_samples)
print("\nMEMORY TEST OF BASE: \n-A:",base_a_val / seed_samples,"\n-B:",base_b_val / seed_samples,"\n-AA:",base_aa_val / seed_samples)
print("\nXOR TEST OF TREE:",test_score)
print("XOR TEST OF BASE:",base_score)

0|1|2|3|4|5|6|7|8|9|10|11|12|13|14|15|16|17|18|19|20|21|22|23|24|25|26|27|28|29|30|31|32|33|34|35|36|37|38|39|40|41|42|43|44|45|46|47|48|49|50|51|52|53|54|55|56|57|58|59|60|61|62|63|64|65|66|67|68|69|70|71|72|73|74|75|76|77|78|79|80|81|82|83|84|85|86|87|88|89|90|91|92|93|94|95|96|97|98|99|

 2.4979206697744814 
TIMES MORE EFFICIENT ON A/B

MEMORY TEST OF TREE: 
-A: 0.04359342916070133 
-B: 0.06589972361217836 
-AA: 0.13426674000366054

MEMORY TEST OF BASE: 
-A: 1.6639089652323853e-07 
-B: 1.0854556154417386e-08 
-AA: 0.49999993600710946

XOR TEST OF TREE: 100
XOR TEST OF BASE: 0
